## Dallas Traffic Crash Data Preprocessing (2021–2025)

### Objective
This notebook cleans and combines annual TxDOT Crash Records Information System (CRIS) exports for crashes within the City of Dallas from 2021 through 2025. It standardizes the source fields, derives spatial and temporal attributes, classifies crash severity, and produces analysis-ready CSV files.

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd

### Data Import and Initial Quality Check

Here, we read the five annual crash records and add `source_year` as a provenance field. Before combining the files, we also summarize their dimensions, duplicate crash IDs, and missing coordinates to catch year-specific issues early.

In [4]:
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    default_data_root = Path(
        "/content/drive/MyDrive/dallas-traffic-saftey-analysis-data/"
    )
else:
    default_data_root = Path("../data").resolve()

data_root = Path(
    os.getenv("DALLAS_CRASH_DATA_ROOT", default_data_root)
)

raw_data = data_root / "01_raw"
processed_data = data_root / "02_processed"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
years = range(2021, 2026)
crash_dfs = {}
expected_columns = None

for year in years:
    file_name = f"dallas-city-{year}-cris.csv"
    file_path = f"{raw_data}/{file_name}"

    # TxDOT CRIS exports include an 11-row metadata preamble before the header
    df = pd.read_csv(file_path, skiprows=11, na_values=["No Data"])

    if expected_columns is None:
        expected_columns = df.columns.tolist()
    else:
        assert df.columns.tolist() == expected_columns, (
            f"{year} columns do not match the first annual export."
        )

    df["source_year"] = year
    crash_dfs[year] = df

In [6]:
qa_metrics = []

for year, df in crash_dfs.items():
    rows, cols = df.shape
    duplicate_ids = df['Crash ID'].duplicated().sum()

    # count records that cannot be represented as crash points
    missing_coords = df[['Latitude', 'Longitude']].isna().any(axis=1).sum()

    qa_metrics.append({
        'Year': year,
        'Rows': rows,
        'Columns': cols,
        'Duplicate IDs': duplicate_ids,
        'Missing coordinates': missing_coords
    })

qa_df = pd.DataFrame(qa_metrics)
display(qa_df)

,Year,Rows,Columns,Duplicate IDs,Missing coordinates
0,2021,37803,22,0,2426
1,2022,36550,22,0,2762
2,2023,34015,22,0,2775
3,2024,27013,22,0,1465
4,2025,25286,22,0,1401


### Merge and Standardize the Crash Records
We append the annual tables into a single dataset, while confirming that no rows were lost during the merge, and standardize the column names for consistent use in Python and ArcGIS.


In [7]:
all_crashes = pd.concat(crash_dfs.values(), ignore_index=True)
all_crashes.head(3)

,Crash ID,City,Contributing Factors,Crash Date,Crash Death Count,Crash Possible Injury Count,Crash Severity,Crash Suspected Serious Injury Count,Crash Time,Crash Total Injury Count,...,Latitude,Light Condition,Longitude,Manner of Collision,Roadway Part,Speed Limit,Street Name,Surface Condition,Weather Condition,source_year
0,18038241,DALLAS,HAD BEEN DRINKING;UNSAFE SPEED,2021-01-01,0,0,B - SUSPECTED MINOR INJURY,0,348,1,...,32.821373,"3 - DARK, LIGHTED",-96.786299,ONE MOTOR VEHICLE - GOING STRAIGHT,1 - MAIN/PROPER LANE,70,US0075,2 - WET,2 - CLOUDY,2021
1,18038267,DALLAS,FAILED TO DRIVE IN SINGLE LANE; HAD BEEN DRINKING,2021-01-01,0,0,N - NOT INJURED,0,319,0,...,32.839214,"3 - DARK, LIGHTED",-96.745393,ONE MOTOR VEHICLE - GOING STRAIGHT,1 - MAIN/PROPER LANE,35,E MOCKINGBIRD LN,2 - WET,2 - CLOUDY,2021
2,18038435,DALLAS,DRIVER INATTENTION,2021-01-01,0,0,99 - UNKNOWN,0,130,0,...,32.687595,"3 - DARK, LIGHTED",-96.814049,ONE MOTOR VEHICLE - GOING STRAIGHT,1 - MAIN/PROPER LANE,30,E PENTAGON PKWY,2 - WET,3 - RAIN,2021


In [8]:
expected_rows = qa_df["Rows"].sum()
assert len(all_crashes) == expected_rows, (
    f"Merged row count is {len(all_crashes):,}; expected {expected_rows:,}."
)
print(f"Merge check passed: {len(all_crashes):,} rows retained.")

Merge check passed: 160,667 rows retained.


In [9]:
all_crashes.columns

Index(['Crash ID', 'City', 'Contributing Factors', 'Crash Date',
       'Crash Death Count', 'Crash Possible Injury Count', 'Crash Severity',
       'Crash Suspected Serious Injury Count', 'Crash Time',
       'Crash Total Injury Count', 'Highway Number', 'Intersection Related',
       'Latitude', 'Light Condition', 'Longitude', 'Manner of Collision',
       'Roadway Part', 'Speed Limit', 'Street Name', 'Surface Condition',
       'Weather Condition', 'source_year'],
      dtype='object')

In [10]:
all_crashes.columns = all_crashes.columns.str.lower().str.replace(' ', '_')
all_crashes.columns

Index(['crash_id', 'city', 'contributing_factors', 'crash_date',
       'crash_death_count', 'crash_possible_injury_count', 'crash_severity',
       'crash_suspected_serious_injury_count', 'crash_time',
       'crash_total_injury_count', 'highway_number', 'intersection_related',
       'latitude', 'light_condition', 'longitude', 'manner_of_collision',
       'roadway_part', 'speed_limit', 'street_name', 'surface_condition',
       'weather_condition', 'source_year'],
      dtype='object')

### Coordinate Cleaning and Spatial Validation
We preserve the source coordinate values before coercing latitude and longitude to numeric fields. Records with both values receive a spatial-eligibility flag, and a broad Dallas-area extent check helps identify obvious coordinate errors without treating the check as an official city-boundary test.


In [11]:
all_crashes["latitude"] = pd.to_numeric(
    all_crashes["latitude"],
    errors="coerce"
)

all_crashes["longitude"] = pd.to_numeric(
    all_crashes["longitude"],
    errors="coerce"
)

In [12]:
all_crashes["has_coordinates"] = (all_crashes["latitude"].notna() &
    all_crashes["longitude"].notna()
)

In [13]:
all_crashes[["latitude", "longitude"]].describe()

,latitude,longitude
count,149838.000000,149838.000000
mean,32.799072,-96.796335
std,0.081579,0.068379
min,32.619817,-96.992710
25%,32.746089,-96.839946
50%,32.791529,-96.802469
75%,32.864527,-96.758643
max,32.987584,-96.517531


In [14]:
all_crashes["has_coordinates"].value_counts()

,count
has_coordinates,
True,149838
False,10829


In [15]:
coordinate_pair_complete = (
    all_crashes["latitude"].notna() == all_crashes["longitude"].notna()
)
assert coordinate_pair_complete.all(), "Some records contain only one coordinate value."

valid_coords = all_crashes[all_crashes["has_coordinates"]]

# screen for obvious coordinate errors using broad Dallas-area sanity bounds
within_dallas_area = (
    valid_coords["latitude"].between(32.0, 33.5) &
    valid_coords["longitude"].between(-97.5, -96.0)
)
assert within_dallas_area.all(), (
    f"{(~within_dallas_area).sum():,} coordinates fall outside the Dallas-area sanity bounds."
)
print("Coordinate validation passed.")

Coordinate validation passed.


### Date Cleaning and Temporal Attributes
Then, we parse the crash date and derive calendar fields used for annual, monthly, weekday, and weekend comparisons. The source year is checked against the parsed year to verify that each record came from the expected annual export.


In [16]:
all_crashes["crash_date"] = pd.to_datetime(
    all_crashes["crash_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

In [17]:
missing_dates = all_crashes["crash_date"].isna().sum()
assert missing_dates == 0, f"{missing_dates:,} crash dates could not be parsed."
print("Date validation passed.")

Date validation passed.


In [18]:
all_crashes["year"] = all_crashes["crash_date"].dt.year
all_crashes["month"] = all_crashes["crash_date"].dt.month
all_crashes["weekday"] = all_crashes["crash_date"].dt.day_name()
all_crashes["month_name"] = all_crashes["crash_date"].dt.month_name()
all_crashes["day_of_week"] = all_crashes["crash_date"].dt.dayofweek
all_crashes["weekend"] = all_crashes["day_of_week"].isin([5, 6]).astype(int)

In [19]:
year_matches_source = all_crashes["year"].eq(all_crashes["source_year"])
assert year_matches_source.all(), (
    f"{(~year_matches_source).sum():,} records do not match their source year."
)
print("Source-year validation passed.")

Source-year validation passed.


### Time Cleaning and Time-Period Classification
We convert the TxDOT crash time from numeric `HHMM` format into separate hour and minute fields, validate the resulting clock values, and assign each crash to a consistent time-of-day period for later temporal analysis.


In [19]:
crash_time_num = pd.to_numeric(
    all_crashes["crash_time"],
    errors="coerce"
)

all_crashes["crash_hour"] = crash_time_num // 100
all_crashes["crash_minute"] = crash_time_num % 100

In [20]:
valid_times = (
    all_crashes["crash_hour"].between(0, 23) &
    all_crashes["crash_minute"].between(0, 59)
)
assert valid_times.all(), f"{(~valid_times).sum():,} records contain invalid crash times."
print("Time validation passed.")

Time validation passed.


In [21]:
def time_period(hour):
    if pd.isna(hour):
        return pd.NA
    elif 0 <= hour < 6:
        return "Overnight"
    elif 6 <= hour < 10:
        return "AM Peak"
    elif 10 <= hour < 16:
        return "Midday"
    elif 16 <= hour < 20:
        return "PM Peak"
    else:
        return "Evening"

all_crashes["time_period"] = all_crashes["crash_hour"].apply(time_period)

### Severity Classification
Here, we extract the TxDOT severity code and create binary indicators for severe and fatal crashes. A severe crash is defined here as a fatal (`K`) or suspected serious-injury (`A`) crash, matching the KABCO injury-severity framework used in the source data.


In [22]:
all_crashes["severity_code"] = (
    all_crashes["crash_severity"]
    .astype("string")
    .str.split(" - ")
    .str[0]
    .str.strip()
)

In [23]:
all_crashes["severity_code"].value_counts(dropna=False)

,count
severity_code,
N,96747
C,24253
B,19486
99,14388
A,4801
K,991
95,1


In [24]:
all_crashes["severe_crash"] = all_crashes["severity_code"].isin(["K", "A"]).astype(int)
all_crashes["fatal_crash"] = (all_crashes["severity_code"] == "K").astype(int)

### Speed Limit Cleaning
TxDOT uses `-1` when the posted speed limit is unknown. We retain those crash records, flag the unknown values, and convert `-1` to a true missing value so it is not analyzed as a physical speed limit.


In [25]:
all_crashes["speed_limit"] = pd.to_numeric(
    all_crashes["speed_limit"],
    errors="coerce"
)

all_crashes["speed_limit_unknown"] = all_crashes["speed_limit"].eq(-1).astype(int)
all_crashes["speed_limit"] = (
    all_crashes["speed_limit"]
    .mask(all_crashes["speed_limit"].eq(-1))
    .astype("Int64")
)

unknown_speed_count = all_crashes["speed_limit_unknown"].sum()
unknown_speed_pct = unknown_speed_count / len(all_crashes) * 100
print(f"Unknown speed limits: {unknown_speed_count:,} ({unknown_speed_pct:.2f}%)")

Unknown speed limits: 8,529 (5.31%)


### Final Validation
Before summarizing and exporting the data, we confirm the core record, location, severity, and speed-limit assumptions used throughout the workflow.


In [26]:
expected_severity_codes = {"K", "A", "B", "C", "N", "95", "99"}

assert all_crashes["crash_id"].is_unique, "Duplicate crash IDs remain after the merge."
assert all_crashes["city"].eq("DALLAS").all(), "The dataset contains records outside Dallas."
assert set(all_crashes["severity_code"].dropna().unique()) <= expected_severity_codes, (
    "Unexpected crash-severity codes were found."
)
assert all_crashes["has_coordinates"].eq(
    all_crashes["latitude"].notna() & all_crashes["longitude"].notna()
).all(), "The coordinate-availability flag is inconsistent."
assert all_crashes["fatal_crash"].eq(all_crashes["crash_death_count"].gt(0)).all(), (
    "Fatal-crash flags do not agree with the recorded death counts."
)
assert all_crashes["speed_limit"].dropna().ge(0).all(), (
    "Negative speed-limit values remain after cleaning."
)

print("All final validation checks passed.")

All final validation checks passed.


### Final Dataset Summary
The following summaries show annual crash outcomes and quantify the records available for spatial analysis. Spatial coverage is reported both by year and for all, severe, and fatal crashes.


In [27]:
annual_summary = all_crashes.groupby("year").agg(
    crashes=("crash_id", "count"),
    spatial_crashes=("has_coordinates", "sum"),
    severe_crashes=("severe_crash", "sum"),
    fatal_crashes=("fatal_crash", "sum")
)
annual_summary["missing_coordinates"] = (
    annual_summary["crashes"] - annual_summary["spatial_crashes"]
)
annual_summary["spatial_coverage_pct"] = (
    annual_summary["spatial_crashes"] / annual_summary["crashes"] * 100
).round(2)

display(annual_summary)

,crashes,spatial_crashes,severe_crashes,fatal_crashes,missing_coordinates,spatial_coverage_pct
year,,,,,,
2021,37803,35377,1277,214,2426,93.58
2022,36550,33788,1206,211,2762,92.44
2023,34015,31240,1148,197,2775,91.84
2024,27013,25548,1109,205,1465,94.58
2025,25286,23885,1052,164,1401,94.46


In [28]:
coverage_groups = {
    "All crashes": pd.Series(True, index=all_crashes.index),
    "Severe crashes": all_crashes["severe_crash"].eq(1),
    "Fatal crashes": all_crashes["fatal_crash"].eq(1),
}

spatial_coverage = pd.DataFrame([
    {
        "crash_group": label,
        "total_crashes": int(mask.sum()),
        "with_coordinates": int(all_crashes.loc[mask, "has_coordinates"].sum()),
    }
    for label, mask in coverage_groups.items()
])
spatial_coverage["missing_coordinates"] = (
    spatial_coverage["total_crashes"] - spatial_coverage["with_coordinates"]
)
spatial_coverage["spatial_coverage_pct"] = (
    spatial_coverage["with_coordinates"] / spatial_coverage["total_crashes"] * 100
).round(2)

display(spatial_coverage.set_index("crash_group"))

,total_crashes,with_coordinates,missing_coordinates,spatial_coverage_pct
crash_group,,,,
All crashes,160667,149838,10829,93.26
Severe crashes,5792,5629,163,97.19
Fatal crashes,991,968,23,97.68


### ArcGIS Field Selection and Export
Finally, we select the fields needed for spatial and temporal analysis, then write two outputs: a spatial dataset limited to records with coordinates and a complete cleaned dataset that retains non-spatial records for tabular analysis and reporting.


In [29]:
arcgis_cols = [
    "crash_id",
    "city",
    "crash_date",
    "year",
    "month",
    "month_name",
    "weekday",
    "day_of_week",
    "crash_time",
    "crash_hour",
    "crash_minute",
    "weekend",
    "time_period",
    "latitude",
    "longitude",
    "has_coordinates",
    "crash_severity",
    "severity_code",
    "severe_crash",
    "fatal_crash",
    "crash_death_count",
    "crash_suspected_serious_injury_count",
    "crash_possible_injury_count",
    "crash_total_injury_count",
    "street_name",
    "highway_number",
    "intersection_related",
    "roadway_part",
    "speed_limit",
    "speed_limit_unknown",
    "manner_of_collision",
    "contributing_factors",
    "light_condition",
    "surface_condition",
    "weather_condition",
    "source_year"
]

df_arcgis = all_crashes[arcgis_cols].copy()

In [30]:
# spatial export: only records that can be mapped as points
df_spatial = df_arcgis[df_arcgis["has_coordinates"]].copy()

df_spatial.to_csv(
    "dallas_crashes_2021_2025_arcgis.csv",
    index=False
)

In [31]:
# complete export: retain records without usable coordinates
df_arcgis.to_csv(
    "dallas_crashes_2021_2025_clean.csv",
    index=False
)